# MedScope AI — Full System on Colab (GPU / CPU)

This notebook clones the repo, loads trained weights from Google Drive, and launches the full stack (FastAPI backend + Next.js frontend) via **ngrok** tunnels.  
No training is performed — use `MedScope_AI_Colab_Training.ipynb` for that.

> **Clinical safety:** This is research/development tooling, not a validated medical device.  
> Do not use generated predictions or reports for patient care without appropriate clinical validation, regulatory review, security controls, and qualified human review.

**Runtime requirements:** GPU (T4 / A100) or CPU · High-RAM recommended · Python 3.10+

### Run order
1. Runtime & Configuration
2. Mount Google Drive
3. Clone Repository
4. Install Dependencies (system → Python → frontend npm)
5. Load Weights from Drive (+ SAM2 baseline auto-download)
6. Configure ngrok
7. Start Infrastructure (Redis · PostgreSQL · Qdrant)
8. Run Database Migrations
9. Start Celery Worker
10. Start Backend API + ngrok tunnel
11. Build & Start Frontend + ngrok tunnel
12. Print Public URLs
13. (Optional) Smoke Tests
14. Keep-Alive loop
15. Shutdown

In [ ]:
#@title 1. Runtime & Configuration { display-mode: "form" }

import os, sys, platform, time
from pathlib import Path

# ── Repository ───────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/prey801/finalyearproject.git"   #@param {type:"string"}
BRANCH      = "main"                                               #@param {type:"string"}
PROJECT_DIR = "/content/finalyearproject"                         #@param {type:"string"}

# ── Google Drive ─────────────────────────────────────────────────────────────
USE_DRIVE         = True                                           #@param {type:"boolean"}
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/medscope-ai/weights"  #@param {type:"string"}

# ── Credentials (fill in here or store as Colab Secrets) ─────────────────────
NGROK_AUTH_TOKEN  = ""  #@param {type:"string"}
SUPABASE_URL      = ""  #@param {type:"string"}
SUPABASE_ANON_KEY = ""  #@param {type:"string"}
GITHUB_TOKEN      = ""  #@param {type:"string"}
OPENAI_API_KEY    = ""  #@param {type:"string"}

# ── Weight filenames (must match what Training notebook saved to Drive) ───────
YOLO_WEIGHT_FILE    = "yolov11_malaria_best.pt"   #@param {type:"string"}
SWIN_WEIGHT_FILE    = "swin_malaria_best.pth"     #@param {type:"string"}
QUALITY_WEIGHT_FILE = "efficientnet_quality.pth"  #@param {type:"string"}
SAM2_WEIGHT_FILE    = "sam2_hiera_small.pt"       #@param {type:"string"}

# ── Colab Secrets fallback ────────────────────────────────────────────────────
def _secret(name, fallback=""):
    if fallback:
        return fallback
    try:
        from google.colab import userdata
        val = userdata.get(name)
        return val if val else fallback
    except Exception:
        return fallback

NGROK_AUTH_TOKEN  = _secret("NGROK_AUTH_TOKEN",  NGROK_AUTH_TOKEN)
SUPABASE_URL      = _secret("SUPABASE_URL",       SUPABASE_URL)
SUPABASE_ANON_KEY = _secret("SUPABASE_ANON_KEY",  SUPABASE_ANON_KEY)
GITHUB_TOKEN      = _secret("GITHUB_TOKEN",       GITHUB_TOKEN)
OPENAI_API_KEY    = _secret("OPENAI_API_KEY",     OPENAI_API_KEY)

# ── Device detection ─────────────────────────────────────────────────────────
try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception:
    DEVICE = "cpu"
    print("PyTorch not yet installed — will be installed in step 4.")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PROJECT_DIR:", PROJECT_DIR)
print("\nConfiguration complete ✓")

In [ ]:
#@title 2. Mount Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive mounted ✓")
else:
    print("Skipping Drive mount (USE_DRIVE=False).")
    print("Weights must be uploaded manually or the backend will run in stub mode.")

In [ ]:
#@title 3. Clone or Update Repository

# Inject GitHub token for private repos
_repo_url = REPO_URL
if GITHUB_TOKEN:
    _repo_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if not Path(PROJECT_DIR).exists():
    !git clone --branch "$BRANCH" "$_repo_url" "$PROJECT_DIR"
else:
    %cd $PROJECT_DIR
    !git fetch origin
    !git checkout "$BRANCH"
    !git pull --ff-only origin "$BRANCH"

%cd $PROJECT_DIR
!git log --oneline -5

# ── Fix: rag/ directory has no __init__.py — create them so Python can import rag.*
from pathlib import Path
for _pkg in [
    "rag",
    "rag/embeddings",
    "rag/vector_store",
    "rag/ingestion",
]:
    _init = Path(PROJECT_DIR) / _pkg / "__init__.py"
    _init.parent.mkdir(parents=True, exist_ok=True)
    if not _init.exists():
        _init.touch()
        print(f"Created {_init}")
    else:
        print(f"OK: {_init}")

print("Repository ready ✓")


In [ ]:
#@title 4. Install System & Python Dependencies

# ── System packages ───────────────────────────────────────────────────────────
print("[1/6] System packages...")
!apt-get update -qq
!apt-get install -y -qq libgl1 redis-server postgresql curl wget

# ── Node.js 20 (for Next.js) ──────────────────────────────────────────────────
print("[2/6] Node.js...")
import subprocess
_node = subprocess.run(["node", "--version"], capture_output=True, text=True)
if _node.returncode != 0 or not _node.stdout.strip().startswith("v"):
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
    !apt-get install -y -qq nodejs
!node --version && npm --version

# ── Python packages ───────────────────────────────────────────────────────────
print("[3/6] Python pip...")
!python -m pip install -q -U pip setuptools wheel

print("[4/6] Pinned boto versions (prevents resolver loops)...")
!python -m pip install -q "aiobotocore==2.13.0" "boto3==1.34.106" "botocore==1.34.106"

print("[5/6] Backend requirements...")
%cd $PROJECT_DIR
!python -m pip install -q -r backend/requirements.txt

print("[6/6] Models requirements (sam2 installed from GitHub source)...")
# sam2 is NOT on PyPI — install from GitHub before the requirements.txt
# so the 'sam2' line in models/requirements.txt resolves correctly.
!python -m pip install -q "git+https://github.com/facebookresearch/sam2.git" || \
    echo "  ⚠  SAM2 GitHub install failed — segmentation will run in stub mode."
# Install remaining models deps, skipping sam2 which is already handled above
!grep -v '^sam2$' models/requirements.txt > /tmp/models_req_filtered.txt
!python -m pip install -q -r /tmp/models_req_filtered.txt
!python -m pip install -q pytest qdrant-client pyngrok nest_asyncio roboflow ultralytics timm

import torch
print("\nPyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("All Python dependencies installed ✓")


In [ ]:
#@title 4b. Install Frontend (npm) Dependencies

%cd $PROJECT_DIR/frontend
!npm ci --prefer-offline 2>&1 | tail -5
print("Frontend npm dependencies installed ✓")

In [ ]:
#@title 5. Load Trained Weights from Drive  (+ SAM2 baseline auto-download)

import shutil
from pathlib import Path

dst_dir = Path(PROJECT_DIR) / "models" / "weights"
dst_dir.mkdir(parents=True, exist_ok=True)

# ── Copy trained weights from Drive ──────────────────────────────────────────
if USE_DRIVE:
    src_dir = Path(DRIVE_WEIGHTS_DIR)
    if src_dir.exists():
        copied = []
        for p in src_dir.glob("*"):
            if p.suffix in {".pt", ".pth", ".ckpt", ".pkl", ".bin"}:
                dst = dst_dir / p.name
                shutil.copy2(p, dst)
                copied.append(p.name)
                print(f"  Copied: {p.name}  ({dst.stat().st_size / 1e6:.1f} MB)")
        if not copied:
            print("  ⚠  No weight files found in Drive.")
            print("     The backend will run in stub/fallback mode.")
        else:
            print(f"\nWeights loaded from Drive ✓  ({len(copied)} files)")
    else:
        print(f"  ⚠  Drive weights directory not found: {src_dir}")
        print("     Backend will run in stub/fallback mode.")
else:
    print("USE_DRIVE=False — skipping Drive copy.")

# ── Download SAM2 baseline weights if missing ─────────────────────────────────
sam2_dst = dst_dir / SAM2_WEIGHT_FILE
if not sam2_dst.exists():
    print("\nDownloading SAM2 hiera-small baseline weights (~180 MB)...")
    %cd $PROJECT_DIR
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt \
        -O "{str(sam2_dst)}"
    print("SAM2 weights downloaded ✓")
else:
    print(f"\nSAM2 weights already present ✓  ({sam2_dst.stat().st_size / 1e6:.1f} MB)")

# ── Inventory ─────────────────────────────────────────────────────────────────
print("\n── Weight inventory ──────────────────────────────────")
for p in sorted(dst_dir.glob("*")):
    print(f"  {p.name:<50s}  {p.stat().st_size / 1e6:7.1f} MB")

In [ ]:
#@title 6. Configure ngrok

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "NGROK_AUTH_TOKEN is required.\n"
        "  1. Sign up / log in at https://dashboard.ngrok.com\n"
        "  2. Copy your authtoken from the 'Your Authtoken' page.\n"
        "  3. Paste it into cell 1 or add as a Colab Secret named 'NGROK_AUTH_TOKEN'."
    )

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()  # close any stale tunnels from previous runs
print("ngrok configured ✓")

In [ ]:
#@title 7. Start Infrastructure (Redis · PostgreSQL · Qdrant)

import subprocess, time, urllib.request
from pathlib import Path

%cd $PROJECT_DIR

# ── Redis ─────────────────────────────────────────────────────────────────────
print("[1/3] Starting Redis...")
!service redis-server start
time.sleep(1)
!redis-cli ping

# ── PostgreSQL ────────────────────────────────────────────────────────────────
print("\n[2/3] Starting PostgreSQL...")
!service postgresql start
time.sleep(2)
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';"  2>/dev/null || true
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;"     2>/dev/null || true
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE medscope_db TO medscope;" 2>/dev/null || true
print("PostgreSQL ready ✓")

# ── Qdrant ────────────────────────────────────────────────────────────────────
print("\n[3/3] Starting Qdrant...")
qdrant_bin = Path(PROJECT_DIR) / "qdrant"
if not qdrant_bin.exists():
    print("  Downloading Qdrant v1.9.2 binary (~90 MB)...")
    !wget -q --show-progress \
        https://github.com/qdrant/qdrant/releases/download/v1.9.2/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz
    !rm -f qdrant-x86_64-unknown-linux-gnu.tar.gz

qdrant_proc = subprocess.Popen(
    [str(qdrant_bin)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    cwd=PROJECT_DIR,
)
time.sleep(5)
try:
    with urllib.request.urlopen("http://localhost:6333/healthz", timeout=5) as r:
        print(f"Qdrant health: {r.read().decode()[:60].strip()} ✓")
except Exception as e:
    print(f"  ⚠  Qdrant healthcheck: {e}  (it may still be starting)")

print("\nAll infrastructure services started ✓")

In [ ]:
#@title 8. Run Database Migrations (Alembic → SQLAlchemy fallback)

import os

# Export shared env vars used by both migration and the backend
os.environ.update({
    "POSTGRES_USER":     "medscope",
    "POSTGRES_PASSWORD": "password",
    "POSTGRES_DB":       "medscope_db",
    "POSTGRES_HOST":     "localhost",
    "QDRANT_HOST":       "localhost",
    "REDIS_URL":         "redis://localhost:6379/0",
})

# IMPORTANT: alembic.ini lives in backend/, not the project root.
# Running alembic from the wrong directory causes 'No config file found'.
%cd $PROJECT_DIR/backend
result = !python -m alembic upgrade head 2>&1
for line in result:
    print(line)

if any("error" in l.lower() for l in result):
    print("Alembic failed — applying SQLAlchemy create_all fallback...")
    %cd $PROJECT_DIR
    !python -c "
import sys; sys.path.insert(0, '.')
from backend.database.session import engine, Base
import backend.database.models  # ensure models are registered
Base.metadata.create_all(bind=engine)
print('Tables created via SQLAlchemy create_all ✓')
"
else:
    print("Alembic migrations applied ✓")

%cd $PROJECT_DIR


## (Optional) Fine-tune SAM2 on Malaria Data

Run the two cells below **before** starting the backend if you want a malaria-specific SAM2 checkpoint.
Skip them if you are happy using the generic pre-trained weights.

**What trains:** mask decoder + prompt encoder only (image encoder is frozen).
**Time:** ~20 min on T4 GPU for 10 epochs.
**VRAM:** ~8 GB peak.

In [ ]:
#@title (Optional) Fine-tune SAM2 mask decoder on malaria dataset

# ── Config ────────────────────────────────────────────────────────────────
SAM2_EPOCHS    = 10   #@param {type:"integer"}
SAM2_BATCH     = 4    #@param {type:"integer"}
SAM2_LR        = 1e-4 #@param {type:"number"}
SAM2_OUT_NAME  = "sam2_malaria_finetuned.pt"  #@param {type:"string"}

import os, subprocess
from pathlib import Path

# Paths
data_dir     = f"{PROJECT_DIR}/data/raw/roboflow_malaria"
base_ckpt    = f"{PROJECT_DIR}/models/weights/{SAM2_WEIGHT_FILE}"
out_ckpt     = f"{PROJECT_DIR}/models/weights/{SAM2_OUT_NAME}"

# Verify Roboflow dataset exists
if not Path(data_dir).exists():
    raise RuntimeError(
        f"Roboflow dataset not found at {data_dir}.\n"
        "Run the download cell in MedScope_AI_Colab_Training.ipynb first, "
        "then re-run this cell."
    )

print("Starting SAM2 fine-tuning...")
print(f"  Data:    {data_dir}")
print(f"  Base:    {base_ckpt}")
print(f"  Output:  {out_ckpt}")
print()

%cd $PROJECT_DIR
!python -m scripts.train_sam2 \
    --data  "$data_dir" \
    --base  "$base_ckpt" \
    --out   "$out_ckpt" \
    --epochs $SAM2_EPOCHS \
    --batch  $SAM2_BATCH \
    --lr     $SAM2_LR \
    --device cuda \
    --drive  "$DRIVE_WEIGHTS_DIR"

# Update the weight file variable so cell 10 (backend) picks it up
if Path(out_ckpt).exists():
    SAM2_WEIGHT_FILE = SAM2_OUT_NAME
    print(f"\n✓ Fine-tuned SAM2 will be used: {SAM2_WEIGHT_FILE}")
else:
    print("\n⚠  Fine-tuning output not found — backend will use base SAM2 weights.")


In [ ]:
#@title 9. Start Celery Worker (async task queue)

import subprocess, os, time

%cd $PROJECT_DIR

celery_env = os.environ.copy()
celery_env["PYTHONPATH"] = PROJECT_DIR

celery_log = open("/tmp/celery.log", "w")
celery_proc = subprocess.Popen(
    ["celery", "-A", "backend.worker", "worker", "--pool=solo", "--loglevel=info"],
    env=celery_env,
    cwd=PROJECT_DIR,
    stdout=celery_log,
    stderr=celery_log,
)
time.sleep(3)
if celery_proc.poll() is not None:
    print("⚠  Celery worker exited early. Check /tmp/celery.log")
else:
    print(f"Celery worker started (PID {celery_proc.pid}) ✓")
    print("Log → /tmp/celery.log")

In [ ]:
#@title 10. Start FastAPI Backend + Expose via ngrok

import subprocess, os, time, urllib.request
from pathlib import Path
from pyngrok import ngrok

%cd $PROJECT_DIR

weights_dir = Path(PROJECT_DIR) / "models" / "weights"

def _wpath(fname):
    p = weights_dir / fname
    return str(p) if p.exists() else ""

backend_env = os.environ.copy()
backend_env.update({
    "PYTHONPATH":           PROJECT_DIR,
    "PROJECT_DIR":          PROJECT_DIR,
    "POSTGRES_USER":        "medscope",
    "POSTGRES_PASSWORD":    "password",
    "POSTGRES_DB":          "medscope_db",
    "POSTGRES_HOST":        "localhost",
    "QDRANT_HOST":          "localhost",
    "REDIS_URL":            "redis://localhost:6379/0",
    "MLFLOW_TRACKING_URI":  "sqlite:///mlflow.db",
    # Trained model weights
    "YOLO_WEIGHTS_PATH":    _wpath(YOLO_WEIGHT_FILE),
    "SWIN_WEIGHTS_PATH":    _wpath(SWIN_WEIGHT_FILE),
    "QUALITY_WEIGHTS_PATH": _wpath(QUALITY_WEIGHT_FILE),
    "SAM2_WEIGHTS_PATH":    _wpath(SAM2_WEIGHT_FILE),
})

# NOTE: ClinicalLLMModel (models/llm/model.py) reads GITHUB_TOKEN, not OPENAI_API_KEY.
# It uses the GitHub Models API (models.github.ai/inference) authenticated by a GitHub PAT.
# Pass GITHUB_TOKEN so the LLM generates real reports instead of stub responses.
if GITHUB_TOKEN:      backend_env["GITHUB_TOKEN"]      = GITHUB_TOKEN
if SUPABASE_URL:      backend_env["SUPABASE_URL"]      = SUPABASE_URL
if SUPABASE_ANON_KEY: backend_env["SUPABASE_ANON_KEY"] = SUPABASE_ANON_KEY

backend_log = open("/tmp/backend.log", "w")
backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    env=backend_env,
    cwd=PROJECT_DIR,
    stdout=backend_log,
    stderr=backend_log,
)

print("Waiting for backend", end="")
for _ in range(30):
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print(" TIMEOUT")
    raise RuntimeError("Backend startup timed out. Check /tmp/backend.log.")

print(" ready!")

backend_tunnel = ngrok.connect(8000, bind_tls=True)
BACKEND_URL = backend_tunnel.public_url
print(f"\n⚙️  Backend API:   {BACKEND_URL}")
print(f"📚 API Docs:      {BACKEND_URL}/docs")
print(f"📊 Metrics:       {BACKEND_URL}/metrics")
print(f"🔍 Log:           /tmp/backend.log")


In [ ]:
#@title 11. Build & Start Next.js Frontend + Expose via ngrok

import subprocess, os, time, urllib.request
from pyngrok import ngrok

frontend_dir = f"{PROJECT_DIR}/frontend"

# Write .env.local so the frontend knows the live backend URL
env_content = f"NEXT_PUBLIC_API_URL={BACKEND_URL}\n"
if SUPABASE_URL:
    env_content += (
        f"NEXT_PUBLIC_SUPABASE_URL={SUPABASE_URL}\n"
        f"NEXT_PUBLIC_SUPABASE_ANON_KEY={SUPABASE_ANON_KEY}\n"
        f"NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY={SUPABASE_ANON_KEY}\n"
    )
with open(f"{frontend_dir}/.env.local", "w") as fh:
    fh.write(env_content)
print(".env.local written ✓")

# Production build
print("\nBuilding Next.js production bundle (may take 2–5 min)...")
%cd $frontend_dir
build = subprocess.run(["npm", "run", "build"], cwd=frontend_dir,
                       capture_output=True, text=True)
if build.returncode != 0:
    print(build.stdout[-2000:])
    print(build.stderr[-2000:])
    raise RuntimeError("Next.js build failed. See output above.")
print("Build succeeded ✓")

# Start production server
frontend_env = os.environ.copy()
frontend_env["PORT"] = "3000"
frontend_log = open("/tmp/frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "start"],
    env=frontend_env,
    cwd=frontend_dir,
    stdout=frontend_log,
    stderr=frontend_log,
)

print("Waiting for frontend", end="")
for _ in range(20):
    try:
        with urllib.request.urlopen("http://localhost:3000", timeout=2) as r:
            if r.status in (200, 307, 308):
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(3)
else:
    print(" (still starting — check /tmp/frontend.log if it takes too long)")

print(" ready!")

frontend_tunnel = ngrok.connect(3000, bind_tls=True)
FRONTEND_URL = frontend_tunnel.public_url
print(f"\n🌍 Frontend:      {FRONTEND_URL}")
print(f"🔍 Log:           /tmp/frontend.log")

In [ ]:
#@title 12. 🚀 System Live — Public URLs

from pyngrok import ngrok as _ng

_tunnels = {t.config["addr"]: t.public_url for t in _ng.get_tunnels()}
_b = _tunnels.get("http://localhost:8000", globals().get("BACKEND_URL",  "N/A"))
_f = _tunnels.get("http://localhost:3000", globals().get("FRONTEND_URL", "N/A"))

print("=" * 65)
print("  🚀  MedScope AI — Full System is Running!")
print("=" * 65)
print(f"  🌍 Web Application:    {_f}")
print(f"  ⚙️  Backend API:        {_b}")
print(f"  📚 API Documentation:  {_b}/docs")
print(f"  📊 Prometheus Metrics: {_b}/metrics")
print("=" * 65)
print()
print("Logs:  /tmp/backend.log  |  /tmp/frontend.log  |  /tmp/celery.log")
print("Run cell 13 to execute smoke tests.")
print("Run cell 14 to keep the system alive.")
print("Run cell 15 to shut everything down.")

In [ ]:
#@title 13. (Optional) Smoke Tests — Health & pytest

import urllib.request, json

_api = "http://localhost:8000"

def _get(path, label):
    try:
        with urllib.request.urlopen(f"{_api}{path}", timeout=10) as r:
            data = json.loads(r.read())
            print(f"  ✅ {label}: {data}")
    except Exception as e:
        print(f"  ❌ {label}: {e}")

print("── API health checks ──")
_get("/",       "Root")
_get("/health", "Health")

print("\n── Model unit tests (mocked, no GPU required) ──")
%cd $PROJECT_DIR
!TESTING=true python -m pytest tests/test_primary_models.py -q --tb=short 2>&1 | tail -25

print("\n── RAG ingestion tests ──")
!TESTING=true python -m pytest tests/test_rag_ingestion.py -q --tb=short 2>&1 | tail -20

print("\nSmoke tests complete ✓")

## ⏳ Keep the System Running

Run the cell below to prevent Colab from disconnecting by actively checking process liveness every minute.  
Press **⏹ Stop** to interrupt it, then run the **Shutdown** cell to cleanly terminate everything.

In [ ]:
#@title 14. Keep-Alive Loop (run until manually interrupted)

import time

print("System is running. Interrupt (Ctrl+C / ⏹) to stop, then run cell 15.\n")

_procs = [
    ("backend",  globals().get("backend_proc")),
    ("frontend", globals().get("frontend_proc")),
    ("celery",   globals().get("celery_proc")),
    ("qdrant",   globals().get("qdrant_proc")),
]

try:
    tick = 0
    while True:
        time.sleep(60)
        tick += 1
        parts = [f"{n}={'✓' if p and p.poll() is None else '✗'}" for n, p in _procs if p]
        print(f"[{tick:4d} min] {' | '.join(parts)}")
except KeyboardInterrupt:
    print("\nInterrupted. Run cell 15 to terminate all processes.")

In [ ]:
#@title 15. Shutdown — Terminate All Processes & Close Tunnels

import subprocess
from pyngrok import ngrok

def _stop(proc, name):
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"  {name}: stopped")
    else:
        print(f"  {name}: already stopped")

print("Stopping application processes...")
_stop(globals().get("frontend_proc"), "Frontend")
_stop(globals().get("backend_proc"),  "Backend")
_stop(globals().get("celery_proc"),   "Celery worker")
_stop(globals().get("qdrant_proc"),   "Qdrant")

print("\nClosing ngrok tunnels...")
ngrok.kill()

print("\nStopping system services...")
!service redis-server stop  2>/dev/null || true
!service postgresql stop    2>/dev/null || true

print("\nAll processes terminated ✓")